# Week 4

Track-A Overture footprints in ten US downtown boxes, mapped to AlphaEarth 10 m cells, then pooled to 100 m tiles. Labels: `resi_def_v1`. Evaluation: leave-one-region-out (LORO).

Polygons are unstratified this week. Older Track-A draws capped each Overture class (~80/class) and ended ~27% residential. The unstratified draw keeps the natural mix (~90% residential among typed buildings, n=492,044). Training balance comes from undersampling residential *pixels*, not from oversampling rare non-residential polygons.

| Head | Metric | Value |
|---|---|---:|
| Logistic regression | macro-F1 (OOF) | 0.806 |
| Logistic regression | LORO mean ± std | 0.779 ± 0.085 |
| Ridge (3 fractions) | mean MAE | 0.254 |
| Ridge | MAE res / nres / empty | 0.223 / 0.251 / 0.287 |


## Data

### Fraction ground truth at 10 m

For each AlphaEarth cell that intersects at least one Track-A footprint:

```text
frac_res   = area(union(RES footprints)  ∩ cell) / area(cell)
frac_nres  = area(union(NRES footprints) ∩ cell) / area(cell)
frac_empty = clip(1 − frac_res − frac_nres, 0, 1)
```

3,151,005 cells. Mean coverage: residential 0.288, non-residential 0.198, empty 0.515. Spec: [`docs/week4/pixel_eval_gt_design.md`](../../docs/week4/pixel_eval_gt_design.md).

![Coverage histograms](../../datasets/week4_pixel_gt/figures/frac_histograms.png)

![Mean fractions by metro](../../datasets/week4_pixel_gt/figures/frac_by_region.png)

### Pure-cell balance

A cell is pure residential if `frac_res > 0` and `frac_nres == 0` (swap for non-residential). Before balance: 2,162,534 RES / 965,515 NRES. Per metro, keep all NRES and draw RES down to match (seed 42) → 1,751,814 cells, 875,907 per class.

![Pure counts before balance](../../datasets/week4_pixel_gt_balanced/figures/imbalanced_counts_by_region.png)

![Pure counts after balance](../../datasets/week4_pixel_gt_balanced/figures/balanced_counts_by_region.png)

### Why the old GT looked NRES-heavy

Stratified Track-A oversampled rare classes. Unstratified Overture in the same AOIs is ~90% residential by building count (~59% by typed area). Global GHSL / UN DESA built-up surface is also ~90% residential. Notebook: [`06_res_nres_eda.ipynb`](06_res_nres_eda.ipynb).

![Building mix: legacy stratified vs unstratified](../../outputs/week4_res_nres_eda/figures/global_vs_stratified_building_mix.png)


## Models at 10 m

Same LORO folds on the balanced pure cells. Features: AlphaEarth `A00`–`A63`. No SMOTE.

1. Ridge multi-output on `(frac_res, frac_nres, frac_empty)`.
2. Logistic regression on the binary stratum label.

Held-out fold models also score every pure cell in the full GT (3,128,049 cells; F1 0.820) so 100 m tiles have dense coverage.

![Fraction true vs predicted](../../outputs/week4_pixel_balanced_train/figures/oof_true_vs_pred.png)

![Fraction MAE by metro](../../outputs/week4_pixel_balanced_train/figures/loro_mae_by_region.png)

![Binary F1 by metro](../../outputs/week4_pixel_balanced_train/figures/loro_f1_by_region.png)

![Binary confusion](../../outputs/week4_pixel_balanced_train/figures/oof_confusion.png)


## Region UMAP (Week 2 layout)

Same ten metros as the Week 2 CNN panel. Blue/red wash = 2D logistic `P(residential)`; black curve = `LR P=0.5`. Numbers below are *in-UMAP* linear separability, not 64-D LORO F1.

Week 2 = CNN on buildings. Week 4 = AlphaEarth on balanced pure 10 m cells (n=59,596). Mean in-UMAP F1: CNN 0.76, AE GT 0.69. Mean share of AE pixels on the wrong side of the cut: 31% (NYC_NE 43%, Phoenix_SW 20%).

Grey = correct side; blue/orange = true class of wrong-side pixels. Panel titles print `% wrong` on the full metro; scatter is stratified to ≤400/metro.

![Wrong-side AE pixels by metro](../../outputs/week4_region_umap/figures/umap_gt_wrong_side_by_region.png)

![Wrong-side rate by metro](../../outputs/week4_region_umap/figures/umap_gt_wrong_side_rate_by_region.png)

Week 2 CNN reference (building patches, same panel order):

![Week 2 CNN UMAP by region + LR boundary](../../outputs/week2_cnn_bbox/umap_cnn_by_region_panels_residential_boundary.png)

| Region | W2 CNN F1 | W4 AE GT F1 | W4 % wrong |
|---|---:|---:|---:|
| Atlanta_SE | 0.71 | 0.69 | 31% |
| Boston_NE | 0.82 | 0.66 | 34% |
| Chicago_MW | 0.79 | 0.70 | 30% |
| Denver_W | 0.76 | 0.67 | 33% |
| Houston_S | 0.77 | 0.67 | 33% |
| LA_W | 0.67 | 0.75 | 25% |
| NYC_NE | 0.70 | 0.55 | 43% |
| Phoenix_SW | 0.77 | 0.80 | 20% |
| SF_W | 0.82 | 0.70 | 30% |
| Seattle_NW | 0.79 | 0.71 | 29% |
| mean | 0.76 | 0.69 | 31% |

Dense GT-colored Week 4 panels (overplots; use wrong-side figure for counts): `outputs/week4_region_umap/figures/umap_gt_by_region_panels_residential_boundary.png`. Rebuild: `scripts/plot_week4_region_umap_boundary.py`.


## Region UMAP 3D

Same 59,596 pixels, UMAP with 3 components, logistic cut is a plane. Grey = correct side; blue/orange = wrong-side by true class. Scatter ≤400/metro; `% wrong` from full n. Mean wrong-side 29% (NYC_NE 40%, Phoenix_SW 21%); mean in-UMAP F1 0.71. Interactive (rotate): [`outputs/week4_region_umap_3d/figures/umap3d_gt_wrong_side_by_region.html`](../../outputs/week4_region_umap_3d/figures/umap3d_gt_wrong_side_by_region.html).

![3D wrong-side AE pixels by metro](../../outputs/week4_region_umap_3d/figures/umap3d_gt_wrong_side_by_region.png)

![3D wrong-side rate by metro](../../outputs/week4_region_umap_3d/figures/umap3d_gt_wrong_side_rate_by_region.png)

| Region | 2D % wrong | 3D % wrong | 2D F1 | 3D F1 |
|---|---:|---:|---:|---:|
| Atlanta_SE | 31% | 31% | 0.69 | 0.69 |
| Boston_NE | 34% | 32% | 0.66 | 0.68 |
| Chicago_MW | 30% | 26% | 0.70 | 0.74 |
| Denver_W | 33% | 33% | 0.67 | 0.67 |
| Houston_S | 33% | 28% | 0.67 | 0.72 |
| LA_W | 25% | 23% | 0.75 | 0.77 |
| NYC_NE | 43% | 40% | 0.55 | 0.60 |
| Phoenix_SW | 20% | 21% | 0.80 | 0.79 |
| SF_W | 30% | 29% | 0.70 | 0.71 |
| Seattle_NW | 29% | 30% | 0.71 | 0.70 |
| mean | 31% | 29% | 0.69 | 0.71 |

Rebuild: `scripts/plot_week4_region_umap_3d.py`.


## 10 m cells vs 100 m tiles

Drop tiles with fewer than five AE 10 m cells (`min ≥ 5`); means and majority votes are unstable on thin tiles. Folders also keep `min ≥ 20`. The three blocks below use different scoring sets; do not mix their numbers.

### Fraction MAE (Ridge)

Full-GT OOF; mean of residential and non-residential MAE. Tile fractions = mean of equal-area 10 m cells.

| | W4 Ridge | GHSL soft share |
|---|---:|---:|
| 10 m | 0.228 | 0.323 |
| 100 m (min ≥ 20) | 0.150 | 0.294 |
| Drop (10 m − 100 m) | 0.079 | 0.030 |

![MAE at 10 m vs 100 m](../../outputs/week4_100m_fraction_eval/figures/mae_10m_vs_100m_ladder.png)

![MAE at 100 m by model](../../outputs/week4_100m_fraction_eval/figures/mae_100m_ladder.png)

### Binary F1 on tiles from balanced cells

Map each balanced train cell to its 100 m tile, keep tiles with ≥ 5 cells, predict with `mean(p_res) ≥ 0.50`.

| | F1 @ 10 m | F1 @ 100 m |
|---|---:|---:|
| W4 LR | 0.806 | 0.817 |
| GHSL | 0.755 | 0.778 |

![LR F1 at 10 m vs 100 m, min ≥ 5](../../outputs/week4_100m_fraction_eval/figures/f1_lr_balanced_10m_vs_100m_min5.png)

### Binary F1 on a 1:1 tile sample

Label eligible tiles by `argmax(frac_res, frac_nres)`, then undersample per metro to equal RES/NRES (52,492 tiles at min ≥ 5). Score with `mean(p_res) ≥ 0.50`.

| | F1 @ τ=0.50 | Best F1 |
|---|---:|---:|
| W4 LR | 0.791 | 0.793 (τ=0.56) |
| GHSL | 0.746 | 0.838 (τ=0.70) |

At the shared operating point τ=0.50, W4 LR beats GHSL. GHSL's best threshold is far from 0.50.

![F1 on 1:1 tiles, min ≥ 5](../../outputs/week4_100m_tiles_balanced/figures/f1_balanced_tiles_ladder_min5.png)

![Threshold sweep on 1:1 tiles](../../outputs/week4_100m_tiles_balanced/figures/f1_balanced_tiles_threshold_curves_min5.png)

![min ≥ 5 support](../../outputs/week4_100m_tiles_balanced/figures/min5_support_explained.png)


## Tile-direct train (10×10 AE grid)

Train on the tile itself: flatten the 10×10 AlphaEarth grid (64-D per cell) plus occupancy mask → 6500-D. Same 1:1 balanced tiles as above. Heads: Ridge on `(frac_res, frac_nres)`, LR on RES vs NRES. LORO over metros. Notebook: [`07_train_100m_tiles.ipynb`](07_train_100m_tiles.ipynb).

### min ≥ 5

| | Tile-direct | Cell-pool `mean(p_res)≥0.50` |
|---|---:|---:|
| OOF F1 | 0.719 | 0.791 |
| LORO F1 mean ± std | 0.700 ± 0.063 | — |
| Mean MAE (frac_res, frac_nres) | 0.448 | — |
| Δ F1 (direct − pool) | -0.071 | |

Linear heads on the flattened grid trail cell-pool here.

![Tile-direct F1 by metro](../../outputs/week4_100m_tile_train/figures/loro_f1_by_region_min5.png)

![Tile-direct MAE by metro](../../outputs/week4_100m_tile_train/figures/loro_mae_by_region_min5.png)

![Tile-direct confusion](../../outputs/week4_100m_tile_train/figures/oof_confusion_min5.png)

### min ≥ 20

| | Tile-direct | Cell-pool @ 0.50 |
|---|---:|---:|
| OOF F1 | 0.726 | 0.808 |
| Mean MAE | 0.401 | — |
| Δ F1 | -0.082 | |

![Tile-direct F1, min≥20](../../outputs/week4_100m_tile_train/figures/loro_f1_by_region_min20.png)


## Paths

| Artifact | Path |
|---|---|
| Fraction GT | `datasets/week4_pixel_gt/` |
| Balanced pure cells | `datasets/week4_pixel_gt_balanced/` |
| Train OOF + checkpoints | `outputs/week4_pixel_balanced_train/` |
| 100 m fraction / LR eval | `outputs/week4_100m_fraction_eval/` |
| 1:1 tile eval | `outputs/week4_100m_tiles_balanced/` |
| Tile-direct train | `outputs/week4_100m_tile_train/` |
| RES/NRES EDA | `outputs/week4_res_nres_eda/` |
| Region UMAP | `outputs/week4_region_umap/` |
| Region UMAP 3D | `outputs/week4_region_umap_3d/` |
